# Forcing files: `_remapped_column` → `_remapped_upper`

**Why:** `rebuild_forcing_bins_20261002` built the radiation and wind `.bin` files
with `FILL_LAYERS = 'column'`, i.e. every segment of a water column received the
surface value. Comparing the `.his` output of the calibrated Baseline
(`his_scenario_files`) with the remapped Baseline (`his_scenario_files_remmaped`)
showed that DELWAQ uses the value given to each segment:

| CAB | calibrated run | `'column'` run |
|---|---|---|
| `RadAve`, layer 8 (top active) | 233.5 | 233.5 |
| `RadAve`, layers 9–16 | 0 | 233.5 |
| `Limit e`, layers 9–16 | 0.99 | mostly 0 |
| phytoplankton, layer 8 / 16 | 0.30 / 0.012 | 2.85 / 2.86 |

(means over 1 Apr – 1 Sep 2023, raw `.his` values; BIS shows the same pattern)

**What this notebook does:** for every `*_remapped_column.bin` in
`wq_0.1min_secondfolder` it writes a `*_remapped_upper.bin` to
`wq_0.1min_secondfolder\forcing_remapped` in which segments in
layers K = 0..`K_FILL_MAX` (default 7) keep their value and deeper segments are 0.
This is the same layer structure the calibrated runs had (compact segments
0–1743 = K 0–6 plus most of K 7), with the horizontal mapping corrected and the
43 previously dark K = 7 cells now included.

The `_remapped_column` files are only read, never changed. Existing outputs are
not replaced unless `OVERWRITE = True`.

**Assumption checked per file:** a `_column` file must have one value per
(M, N) column. Records where that is not true are counted; any non-zero count
marks the file `CHECK` and its output should not be used.

## 1 — Configuration

In [ ]:
import os, glob
import numpy as np
import pandas as pd

BIN_DIR     = r'<MODEL_DIR>'   # <-- set to wq_0.1min_secondfolder
LOOKUP_PATH = os.path.join(BIN_DIR, 'segment_lookup.npz')   # segnum, new_to_old
OUT_DIR     = os.path.join(BIN_DIR, 'forcing_remapped')     # where the _upper files are written
os.makedirs(OUT_DIR, exist_ok=True)

K_FILL_MAX = 7          # layers 0..K_FILL_MAX keep their value, deeper layers -> 0
OVERWRITE  = False      # refuse to replace an existing output file
CHUNK      = 2000       # records processed per chunk
CHECK_RECORD_FRACTION = 0.55   # record used in the checks (≈ summer for daily radiation)

# All scenarios. The Baseline (noFPV) files were converted and verified on 8 Oct 2026:
# their outputs already exist, so they are skipped. Use '*noFPV*_remapped_column.bin'
# to limit the run to the Baseline again.
PATTERN = '*_remapped_column.bin'

in_files = sorted(glob.glob(os.path.join(BIN_DIR, PATTERN)))   # secondfolder only, no subfolders
print(f'{len(in_files)} file(s) matching {PATTERN!r} in {BIN_DIR}:')
for p in in_files:
    done = os.path.exists(os.path.join(OUT_DIR, os.path.basename(p).replace('_remapped_column.bin',
                                                                           '_remapped_upper.bin')))
    print(f'  {os.path.basename(p):<75} {os.path.getsize(p)/1e6:>9,.1f} MB'
          f'{"   [output exists - will be skipped]" if done else ""}')
print(f"\n{sum('radiation' in os.path.basename(p) for p in in_files)} radiation, "
      f"{sum('wind' in os.path.basename(p) for p in in_files)} wind - "
      'compare with the files named in the scenario .inp files.')

## 2 — Layer of every compact segment

Same lookup as `rebuild_forcing_bins_20261002` (section 2): `segnum` gives the
FLOW segment id of each (M, N, K) cell and `new_to_old` the FLOW id of each of
the 10,999 compact WAQ segments.

In [ ]:
lookup = np.load(LOOKUP_PATH)
segnum, new_to_old = lookup['segnum'], lookup['new_to_old'].astype(np.int64)
N_SEG = len(new_to_old)
M, N, K = segnum.shape

valid = ~np.isnan(segnum)
ids = np.where(valid, segnum, 0).astype(np.int64) - 1          # 0-based FLOW id
n_flow = int(ids[valid].max()) + 1

flow_m = np.full(n_flow, -1); flow_n = np.full(n_flow, -1); flow_k = np.full(n_flow, -1)
mm, nn, kk = np.nonzero(valid)
flow_m[ids[mm, nn, kk]] = mm; flow_n[ids[mm, nn, kk]] = nn; flow_k[ids[mm, nn, kk]] = kk

c_m, c_n, LAYER_OF = flow_m[new_to_old], flow_n[new_to_old], flow_k[new_to_old]
assert (c_m >= 0).all(), 'some compact segments are not in segnum'
COL_ID = c_m * N + c_n                     # (M, N) column of every compact segment
KEEP   = LAYER_OF <= K_FILL_MAX

print(f'segnum {segnum.shape}, compact segments N_SEG = {N_SEG}')
print(pd.DataFrame({'segments': np.bincount(LAYER_OF, minlength=K),
                    'kept':     np.bincount(LAYER_OF[KEEP], minlength=K)}
                   ).rename_axis('K').T.to_string())
print(f'\nKept: {KEEP.sum()} segments (K <= {K_FILL_MAX}); set to 0: {(~KEEP).sum()}')

## 3 — Convert

Each record is: timestamp (int32) + `N_SEG` float32 values. Timestamps are
copied unchanged.

In [ ]:
REC_DTYPE = np.dtype([('t', '<i4'), ('v', '<f4', (N_SEG,))])

# One reference segment per column: in a '_column' file every segment must equal it
_ref = np.zeros(COL_ID.max() + 1, np.int64)
_ref[COL_ID] = np.arange(N_SEG)
REF_OF = _ref[COL_ID]

def out_path_for(in_path):
    name = os.path.basename(in_path).replace('_remapped_column.bin', '_remapped_upper.bin')
    return os.path.join(OUT_DIR, name)

def column_to_upper(in_path, out_path):
    size = os.path.getsize(in_path)
    nrec, rem = divmod(size, REC_DTYPE.itemsize)
    if rem or nrec == 0:
        return {'status': f'SKIPPED: size {size:,} is not a whole number of records'}
    if os.path.exists(out_path) and not OVERWRITE:
        out_size = os.path.getsize(out_path)
        if out_size != size:
            return {'status': f'SKIPPED: INCOMPLETE output exists ({out_size:,} of {size:,} bytes) - '
                              'delete it or set OVERWRITE=True'}
        return {'status': 'SKIPPED: output exists (set OVERWRITE=True to rebuild)',
                'records': nrec}
    src = np.memmap(in_path, dtype=REC_DTYPE, mode='r', shape=(nrec,))
    not_column = 0
    with open(out_path, 'wb') as fo:
        for i0 in range(0, nrec, CHUNK):
            blk = src[i0:i0 + CHUNK]
            v = np.array(blk['v'])
            not_column += int((v != v[:, REF_OF]).any(axis=1).sum())
            v[:, ~KEEP] = 0.0
            out = np.empty(len(blk), REC_DTYPE)
            out['t'] = blk['t']
            out['v'] = v
            out.tofile(fo)
    del src
    ok_size = os.path.getsize(out_path) == size
    status = ('OK' if (ok_size and not_column == 0) else
              'CHECK: input not column-uniform' if not_column else 'CHECK: size mismatch')
    return {'status': status, 'records': nrec, 'records not column-uniform': not_column}

results = {}
for p in in_files:
    q = out_path_for(p)
    print(f'{os.path.basename(p)}  →  {os.path.basename(q)}', flush=True)
    r = column_to_upper(p, q)
    r.update(input=p, output=q)
    results[os.path.basename(p)] = r
    print(f'    {r["status"]}', flush=True)

summary = pd.DataFrame(results).T.reindex(columns=['status', 'records', 'records not column-uniform'])
print('\n' + summary.to_string())

## 4 — Checks

For one record per file (same record in input and output):
* **conversion status:** must be `OK` (from section 3);
* **time equal:** timestamps copied unchanged;
* **kept layers unchanged:** values in K ≤ `K_FILL_MAX` identical to the input;
* **deep layers zero:** no non-zero value below `K_FILL_MAX`;
* **non-zero segments per layer:** for daily radiation in summer, expected
  `[104, 161, 45, 37, 25, 36, 22, 1357, 0, …, 0]` (fewer if some cells are fully shaded).

In [ ]:
def read_record(path, rec):
    with open(path, 'rb') as f:
        f.seek(rec * REC_DTYPE.itemsize)
        t = np.frombuffer(f.read(4), '<i4')[0]
        return t, np.frombuffer(f.read(4 * N_SEG), '<f4')

rows = []
for name, r in results.items():
    if not os.path.exists(r.get('output', '')) or 'records' not in r:
        continue
    rec = int(r['records'] * CHECK_RECORD_FRACTION)
    t_in,  v_in  = read_record(r['input'],  rec)
    t_out, v_out = read_record(r['output'], rec)
    rows.append({'file': name, 'conversion status': r['status'], 'record': rec,
                 'time equal': bool(t_in == t_out),
                 'kept layers unchanged': bool(np.array_equal(v_in[KEEP], v_out[KEEP])),
                 'deep layers zero': bool((v_out[~KEEP] == 0).all()),
                 'non-zero segments per layer': np.bincount(LAYER_OF[v_out != 0], minlength=K).tolist()})

pd.set_option('display.max_colwidth', 90)
checks = pd.DataFrame(rows)
print(checks.to_string(index=False))
if len(checks):
    skipped = checks['conversion status'].str.startswith('SKIPPED')
    bad = (checks['conversion status'].str.startswith('CHECK') |
           ~(checks['time equal'] & checks['kept layers unchanged'] & checks['deep layers zero']))
    if bad.any():
        print('\n⚠ Do not use:', ', '.join(checks.loc[bad, 'file']))
    if (skipped & ~bad).any():
        print('\nℹ Not converted in this run (output already existed), so the column-uniformity '
              'check was not repeated:', ', '.join(checks.loc[skipped & ~bad, 'file']))
    if not bad.any():
        n_new = int((~skipped).sum())
        print(f'\n✓ All checked outputs pass ({n_new} converted in this run, '
              f'{int(skipped.sum())} existing).')
incomplete = [n for n, r in results.items() if 'INCOMPLETE' in r.get('status', '')]
if incomplete:
    print('\n⚠ Incomplete output (not checked) - delete it or set OVERWRITE=True:', ', '.join(incomplete))

## 5 — Next steps
1. Point the Baseline `.inp` at the two `forcing_remapped\*_remapped_upper.bin` files
   (radiation and wind), or copy them to the folder the `.inp` reads from.
2. Re-run the Baseline and repeat the `.his` comparison against `his_scenario_files`:
   `RadAve` should be 233.5 in layer 8 and 0 in layers 9–16, and phytoplankton
   should be close to the calibrated run (not identical: 43 surface cells now
   receive light and wind).
3. ✓ Done 8 Oct 2026: the Baseline matched the calibrated run (`RadAve` and
   `Limit e` identical at CAB and BIS; phytoplankton within a few %), so
   `PATTERN` now converts all scenarios.
4. Point each FPV scenario `.inp` at its `forcing_remapped\*_remapped_upper.bin`
   files (radiation and wind) and re-run the scenarios.